In [0]:
%sql
create schema if not exists hon;
use schema hon ;
select current_catalog(),current_schema();

current_catalog(),current_schema()
kishore_d,hon


## Section - 1

In [0]:
#q1 --> using only dbfs - Display where the source datasets are stored in  - MAIN HONS
display(dbutils.fs.ls('/Volumes/kishore_d/hon/csv_files'))

path,name,size,modificationTime
dbfs:/Volumes/kishore_d/hon/csv_files/food_delivery.csv,food_delivery.csv,821562,1786029088000
dbfs:/Volumes/kishore_d/hon/csv_files/food_sales.csv,food_sales.csv,572133,1786029088000
dbfs:/Volumes/kishore_d/hon/csv_files/food_sales1.csv,food_sales1.csv,228505,1786029088000
dbfs:/Volumes/kishore_d/hon/csv_files/food_sales2.csv,food_sales2.csv,343700,1786029088000


In [0]:
# q2 --> show contents using the dbfs   - MAIN HONS
# We can use %fs head /Volumes/kishore_d/hon/csv_files/food_sales.csv to see the contents of the file
print(dbutils.fs.head('/Volumes/kishore_d/hon/csv_files/food_sales.csv', 1000))
# print(dbutils.fs.head('/Volumes/kishore_d/hon/csv_files/food_sales1.csv', 1000))
# print(dbutils.fs.head('/Volumes/kishore_d/hon/csv_files/food_sales2.csv', 1000))
print(dbutils.fs.head('/Volumes/kishore_d/hon/csv_files/food_delivery.csv', 1000))

[Truncated to first 1000 bytes]
OrderID,MenuItem,Quantity,PricePerItem,TotalPrice,Discount,Tax,OrderDate
1000,Item_4,3,19.43,58.29,1.95,0.74,2023-01-01 00:00:00
1001,Item_10,1,11.3,11.3,4.57,1.55,2023-01-01 01:00:00
1002,Item_20,4,10.33,41.32,3.79,1.95,2023-01-01 02:00:00
1003,Item_17,1,14.76,14.76,0.38,2.19,2023-01-01 03:00:00
1004,Item_16,2,6.74,13.48,2.62,2.05,2023-01-01 04:00:00
1005,Item_8,3,17.23,51.69,4.59,1.81,2023-01-01 05:00:00
1006,Item_5,1,15.11,15.11,4.83,1.32,2023-01-01 06:00:00
1007,Item_1,3,7.73,23.19,3.48,0.74,2023-01-01 07:00:00
1008,Item_7,5,10.18,50.9,1.68,1.62,2023-01-01 08:00:00
1009,Item_4,2,6.61,13.22,1.75,0.9,2023-01-01 09:00:00
1010,Item_20,3,6.64,19.919999999999998,3.73,2.17,2023-01-01 10:00:00
1011,Item_1,2,10.68,21.36,4.88,2.38,2023-01-01 11:00:00
1012,Item_2,5,14.81,74.05,2.54,2.38,2023-01-01 12:00:00
1013,Item_9,1,17.85,17.85,3.18,2.06,2023-01-01 13:00:00
1014,Item_19,4,12.1,48.4,2.67,1.68,2023-01-01 14:00:00
1015,Item_16,5,19.1,95.5,4.07,0.54,2023-01-01 

### Spark RDD

In [0]:
# q -- Get orders count where the discount > 3.0 using spark rdd  for food_sales.csv
# from pyspark.sql import SparkSession
# spark = SparkSession.builder.appName("Spark DataFrames").getOrCreate()
# # q3 --> Read the csv files into Spark DataFrames
# sc=spark.sparkContext
# df_rdd=sc.textFile('/content/food_sales.csv')
# header = df_rdd.first()
# res=df_rdd.filter(lambda line: line != header).map(lambda x: x.split(',')).map(lambda x:(x[0],x[5]))
# res1=res.filter(lambda x:float(x[1]) >3.0)
# res1.count()
# using df bcoz we cant use the rdd in serverless
df_rdd_sales=spark.read.csv('/Volumes/kishore_d/hon/csv_files/food_sales.csv',header=True,inferSchema=True)
df_rdd_sales.filter(df_rdd_sales['Discount'] > 3.0).count()

4008

In [0]:
# counting unique customer - MAIN HONS
df_distinct = df_del.dropDuplicates(["CustomerName"]).count()
display(df_distinct)

100

In [0]:
# Finding the most popular payment method - MAIN HONs
df_join=df_rdd_sales.join(df_del,on="OrderID",how="inner")
res=df_join.groupBy('paymentMethod').count()
res.withColumnRenamed('count','No_of_Payments').orderBy('No_of_Payments',ascending=False).limit(1).display()

paymentMethod,No_of_Payments
Online Payment,3414


In [0]:
# spark sql 
# q -- top 5 busiest day for deliveries 
df_del=spark.read.csv('/Volumes/kishore_d/hon/csv_files/food_delivery.csv',header=True,inferSchema=True)
df_del.createOrReplaceTempView('delivery')
spark.sql('select DeliveryDate,count(*) as No_of_deliveries from delivery group by DeliveryDate order by No_of_deliveries desc limit 5').display()

DeliveryDate,No_of_deliveries
2023-02-24,1
2023-02-11,1
2023-02-03,1
2023-02-17,1
2023-03-21,1


In [0]:
# avg deliveries per driver - MAIN HONS
spark.sql('select DriverID,avg(DeliveryID) from delivery group by DriverID order by DriverID').display()

DriverID,avg(DeliveryID)
1,4973.6442307692305
2,4857.695238095238
3,4905.663157894737
4,4956.080459770115
5,5172.689320388349
6,4670.476190476191
7,5442.288888888889
8,5253.795698924731
9,5226.211382113821
10,5051.148148148148


In [0]:
# avg items per order
df_rdd_sales.createOrReplaceTempView('sales')
spark.sql('select OrderID, avg(Quantity) as avg_food_orders from sales group by 1 order by OrderID').display()

OrderID,avg_food_orders
1000,3.0
1001,1.0
1002,4.0
1003,1.0
1004,2.0
1005,3.0
1006,1.0
1007,3.0
1008,5.0
1009,2.0


In [0]:
# Finding nulls in both tables using spark sql - MAIN HONS
spark.sql('select count(*) as cnt from sales where OrderID is null').display()
spark.sql('select count(*) as cnt from delivery where DeliveryID is null').display()

cnt
0


cnt
0


## Section -2

In [0]:
# Create delta table for the food_sales and food_delivery  - MAIN HONS
df_rdd_sales.write.format('delta').mode('append').saveAsTable('food_sales')
df_del.write.format('delta').mode('append').saveAsTable('food_delivery')

In [0]:
%sql
-- how many deliveries done for the each item 
select MenuItem,count(*) as total_item_delivered from food_sales group by MenuItem order by 2 desc ;

MenuItem,total_item_delivered
Item_1,553
Item_18,536
Item_15,529
Item_11,515
Item_12,512
Item_8,511
Item_6,510
Item_14,507
Item_10,500
Item_17,500


In [0]:
spark.sql("""
          select s.OrderID as order_id,d.CustomerName as cust_name,d.DeliveryDate as order_date,s.TotalPrice as total_amount from food_sales s join food_delivery d 
          on s.OrderID=d.OrderID
          """)

DataFrame[order_id: int, cust_name: string, order_date: date, total_amount: double]

In [0]:
%sql
drop table sal_del_dt

In [0]:
%sql
-- Create a delta table using specific cols from both sales and delivery 
create table sal_del_dt 
using delta
as 
select s.OrderID as order_id,d.CustomerName as cust_name,d.DeliveryDate as order_date,s.TotalPrice as total_amount 
from food_sales s join food_delivery d 
          on s.OrderID=d.OrderID ;
select * from sal_del_dt limit 5;

order_id,cust_name,order_date,total_amount
3765,Emily Doe,2023-01-01,35.88
1757,David Brown,2023-01-02,17.12
9415,John Johnson,2023-01-03,79.96
2536,Michael Johnson,2023-01-04,6.96
5347,William Brown,2023-01-05,20.549999999999997


## Section -3

In [0]:
# Autoloader - remove food_sales1,2.csv files - MAIN HONS
source='/Volumes/kishore_d/hon/csv_files/'
dest='/Volumes/kishore_d/hon/schema/'
dest1='/Volumes/kishore_d/hon/schema/chk'
df_stream=(spark.readStream
           .format('cloudFiles')
           .option('cloudFiles.format','csv')
           .option('cloudFiles.schemaLocation', dest)
           .option('pathGlobFilter','food_sales*.csv')
           .option('header','true')
           .option('inferSchema','true')
           .load(source)
           )
(df_stream.writeStream.format('delta')
        .option('checkpointLocation',dest1)
        .outputMode('append')
        .trigger(availableNow=True)
        .toTable('kishore_d.hon.food_sales_streaming')
        )

In [0]:
%sql
select count(*)  as cnt1 from food_sales_streaming;

cnt1
14001


In [0]:
spark.sql('select MenuItem,sum(Quantity) as total_quantity_01 from food_sales_streaming group by MenuItem order by total_quantity_01 desc ').display()

MenuItem,total_quantity_01
Item_1,2352.0
Item_18,2328.0
Item_11,2271.0
Item_15,2238.0
Item_12,2132.0
Item_8,2119.0
Item_2,2079.0
Item_10,2078.0
Item_6,2075.0
Item_14,2060.0


In [0]:
%sql
-- add food_sales1.csv and stream cell again and run this cell to see the new data
select count(*) as cnt2 from food_sales_streaming

cnt2
20000


In [0]:
spark.sql('select MenuItem,sum(Quantity) as total_quantity_02 from food_sales_streaming group by MenuItem order by total_quantity_02 desc ').display()

MenuItem,total_quantity_02
Item_1,3280.0
Item_18,3268.0
Item_15,3212.0
Item_11,3164.0
Item_12,3114.0
Item_10,3044.0
Item_6,2998.0
Item_14,2976.0
Item_13,2966.0
Item_8,2956.0


In [0]:
from pyspark.sql.types import *
source = "/Volumes/kishore_d/hon/csv_files/"
checkpoint_loc = "/Volumes/kishore_d/hon/schema/chk_basic/"

schema= StructType([
    StructField("OrderID", IntegerType(), True),
    StructField("MenuItem", StringType(), True),
    StructField("Quantity", IntegerType(), True),
    StructField("PricePerItem", DoubleType(), True),
    StructField("TotalPrice", DoubleType(), True),
    StructField("Discount", DoubleType(), True),
    StructField("Tax", DoubleType(), True),
    StructField("OrderDate", TimestampType(), True)
])
# Autoloader - remove food_sales1,2.csv files - MAIN HONS



df_stream = (spark.readStream
    .format("csv")                # direct CSV reader
    .option("header", "true")     # use first row as header
        .load(source, schema=schema)   # watch the folder for new files
)

(df_stream.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_loc)
    .outputMode("append")
    .trigger(once=True)           # or continuous if you want
    .toTable("kishore_d.hon.food_sales_streaming_basic")
)


In [0]:
%sql
select count(*) from food_sales_streaming_basic; -- Getting dups and overlapping values

count(*)
24001


In [0]:
%sql
-- History 
describe history kishore_d.hon.food_sales_streaming ; 

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
2,2026-08-06T16:47:57.000Z,144097337326174,fy26databricks5thaugto7thsepuser157@acp11792b.onmicrosoft.com,STREAMING UPDATE,"Map(outputMode -> Append, queryId -> 43590758-e089-4852-8358-785bca7f63ad, epochId -> 1, statsOnLoad -> true)",null,List(3742447464672783),029524a1-f77b-4f45-bd70-f9965a781929,0806-151018-mybvcr5z-v2n,1,WriteSerializable,true,"Map(numRemovedFiles -> 0, numOutputRows -> 4001, numOutputBytes -> 51828, numAddedFiles -> 1)",null,Databricks-Runtime/18.x-photon-scala2.13
1,2026-08-06T16:45:03.000Z,144097337326174,fy26databricks5thaugto7thsepuser157@acp11792b.onmicrosoft.com,STREAMING UPDATE,"Map(outputMode -> Append, queryId -> 43590758-e089-4852-8358-785bca7f63ad, epochId -> 0, statsOnLoad -> true)",null,List(3742447464672783),b6247037-2047-4440-b495-483fab3f4b00,0806-151018-mybvcr5z-v2n,0,WriteSerializable,true,"Map(numRemovedFiles -> 0, numOutputRows -> 10000, numOutputBytes -> 107976, numAddedFiles -> 1)",null,Databricks-Runtime/18.x-photon-scala2.13
0,2026-08-06T16:44:56.000Z,144097337326174,fy26databricks5thaugto7thsepuser157@acp11792b.onmicrosoft.com,CREATE TABLE,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-c401f33e-7882-44c5-a934-0a5eca68bdbd"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-ba2b44e6-28bb-4d9c-bc9c-e50f92cc3e50""}, statsOnLoad -> false)",null,List(3742447464672783),b6247037-2047-4440-b495-483fab3f4b00,0806-151018-mybvcr5z-v2n,null,WriteSerializable,true,Map(),null,Databricks-Runtime/18.x-photon-scala2.13


In [0]:
%sql
-- Time -travel 
select * from kishore_d.hon.food_sales_streaming version as of 2 ; -- 2 is the version number

In [0]:
%sql
-- vacuum 
vacuum kishore_d.hon.food_sales_streaming_basic retain 168 hours dry run  ;
-- 7 days (default by databricks) we get the overrided or overwritted rows data output 

path


In [0]:
%sql
-- optimize  ---> Use OPTIMIZE to compact streaming output files and reduce the number of files in a table
-- OPTIMIZE is a Databricks-specific command that compacts the data files in a Delta table to reduce the number of files and improve query performance.
-- OPTIMIZE is useful for tables that have a large number of small files, which can lead to poor query performance.
optimize kishore_d.hon.food_sales_streaming zorder by (OrderID);
    
-- Z-Ordering 
-- Z-ordering is a technique for optimizing queries on tables with skewed data. 
-- It is a form of indexing that reorders the data in a table based on the values of one or more columns.
-- This can improve the performance of queries that filter or join on those columns.
-- Z-ordering is particularly useful for tables with skewed data, where a small number of values occur much more frequently than others.
-- By reordering the data based on these values, queries can more quickly locate the relevant data and reduce the amount of data that needs to be scanned.
-- In Databricks, you can use the ZORDER BY clause to specify the columns to use for z-ordering.
-- For example, the following command creates a table with z-ordering on the columns col1 and col2:
-- create table my_table (col1 int, col2 int) using delta zorder by (col1, col2);
-- The ZORDER BY clause can also be used with the OPTIMIZE command to compact files and improve query performance.
-- For example, the following command optimizes the table and z-orders the data on the columns col1 and col2:
-- optimize my_table zorder by (col1, col2);


path,metrics
,"List(1, 2, List(132879, 132879, 132879.0, 1, 132879), List(51828, 107976, 79902.0, 2, 159804), 0, List(minCubeSize(107374182400), List(0, 0), List(2, 159804), 0, List(2, 159804), 1, null), null, 0, 1, 2, 0, false, 0, 0, 1786036006973, 1786036009240, 8, 1, null, List(0, 0), null, 9, 9, 339, 0, null, null)"


Key Metrics Explained
numFilesAdded: 1 / numFilesRemoved: 2  
→ OPTIMIZE compacted 2 small files into 1 larger file.

filesAdded  
→ The new compacted file is ~132 KB.

filesRemoved  
→ Two old files (sizes ~51 KB and ~107 KB) were deleted.

zOrderStats  
→ Shows how ZORDER clustering was applied.

strategyName: minCubeSize(107374182400) → ZORDER tries to build “cubes” of data for clustering, but your dataset is small, so it just merged files.

inputOtherFiles: {"num": 2, "size": 159804} → Two files were considered for clustering.

numOutputCubes: 1 → One cube (clustered file) was produced.

partitionsOptimized: 0  
→ No partition pruning was needed (your table isn’t partitioned).

totalTaskExecutionTimeMs: 339  
→ The OPTIMIZE job finished in ~0.3 seconds — very fast because the dataset is small.


What This Means
Your streaming table had small files (typical for streaming ingestion).

OPTIMIZE compacted them into a single larger file.

ZORDER was applied, but since the dataset is tiny, you won’t see dramatic clustering benefits yet.


Best Practice for Streaming Tables
Run OPTIMIZE periodically (e.g., nightly) to compact small files.

ZORDER by frequently queried columns (like OrderID, DeliveryDate).

For large datasets, this will reduce scan time significantly.

For small datasets, you’ll mostly just see file compaction.